In [32]:
import numpy as np
import pandas as pd
from scipy import signal
from scipy.io import wavfile
from scipy.fft import fftshift
import matplotlib.pyplot as plt
import librosa
from scipy.signal import hilbert, butter, lfilter
import os
import textgrid
import sys
import shutil
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import precision_recall_curve
import h5py

In [33]:
# Generate folds based on individuals

X = np.array(range(1,11))
y = np.ones(X.shape)

skf1 = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)

folds = []

for i, (train_index, test_index) in enumerate(skf1.split(X, y)):
    #print(f"Fold {i}:")
    test = X[test_index]

    shift_add = 5 #random.randint(1,9)
    if test_index[1] == (test_index[0] + shift_add) % 10 or test_index[0] == (test_index[1] + shift_add) % 10:
        shift_add = shift_add + 1

    validate_index = (np.array(test_index) + shift_add) % 10
    validate = X[validate_index]

    train_index = [i for i in train_index if i not in validate_index]
    train = X[train_index]

    check = [x for x in X if x in validate and x in test]
    assert(len(check)==0)

    folds.append(np.concatenate([train, test, validate]))

# Save fold into CSV
folds = np.array(folds)
colums = ["tr"+str(i) for i in range(1,7)]
colums.extend(["v"+str(i) for i in range(1,3)])
colums.extend(["t"+str(i) for i in range(1,3)])

pd.DataFrame(data=folds, columns=colums).to_csv('YellowHammer/Training_set_YH_songs/folds.csv', index=False)

In [34]:
# Split negatives 
def negative_folds(folder_negative):
    files = os.listdir(folder_negative)
    files = [f for f in files if ".wav"]
    y = [0 for f in files]

    X_train, X_test, y_train, y_test = train_test_split(files, y, test_size=0.2, random_state=1)

    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=1)

    return X_train, X_val, X_test

In [35]:
def get_files(folder, id_list): # Get files that match individuals in id_list
    files = os.listdir(folder)
    list_files = []
    for id_indiv in id_list:
        yes_files = [f for f in files if "YH_"+str(id_indiv)+"_" in f]
        list_files.extend(yes_files)
    return list_files

In [36]:
def get_slice(list, start, end):
    return list[int(start*len(list)):int(end*len(list))]

### Use folds

In [38]:
neg_train, neg_valid, neg_test = negative_folds('YellowHammer/Training_set_YH_songs/negative/') #common across folds

fld_nb = 0
for fld_nb in range(df_folds.shape[0]):
    fold = df_folds[fld_nb]
    fold_train, fold_valid, fold_test = fold[0:6], fold[6:8], fold[8:10]

    pos_train = get_files(folder_data, fold_train)
    pos_valid = get_files(folder_data, fold_valid)
    pos_test = get_files(folder_data, fold_test)

    # Load files in each fold then you can use them...

### To HDF5

In [26]:
def to_h5(folder, files, size_h5dataset, outname, outfolder):
    nb_files = int(len(files)/size_h5dataset)

    findex = 0
    for i in range (nb_files):
        dataset = []
        for j in range(size_h5dataset):
            song_file = files[findex]
            findex = findex + 1
            samplerate, data = wavfile.read(folder+song_file)
            data = data.astype(float)
            dataset.append(data)
        dataset = np.array(dataset)

        hdf5_file_path = outfolder+outname+'_'+str(i+1)+'.h5'

        with h5py.File(hdf5_file_path, 'w') as hdf5_file:
            hdf5_file.create_dataset('data', data=dataset)
            hdf5_file.close()

In [37]:
folder_data = "YellowHammer/Training_set_YH_songs/positive/"

df_folds = pd.read_csv('YellowHammer/Training_set_YH_songs/folds.csv')
df_folds = df_folds.to_numpy()

In [44]:
neg_train, neg_valid, neg_test = negative_folds('YellowHammer/Training_set_YH_songs/negative/') #common across folds

folder_neg = 'YellowHammer/Training_set_YH_songs/negative/'
outfolder = 'YellowHammer/Training_set_YH_songs/data_h5/'

if True: #not os.path.exists(outfolder):
    #os.makedirs(outfolder)

    to_h5(folder_neg, neg_train, len(neg_train), "neg_train", outfolder)
    to_h5(folder_neg, neg_valid, len(neg_valid), "neg_valid", outfolder)
    to_h5(folder_neg, neg_test, len(neg_test), "neg_test", outfolder)


    for fld_nb in range(df_folds.shape[0]):
        fold = df_folds[fld_nb]
        fold_train, fold_valid, fold_test = fold[0:6], fold[6:8], fold[8:10]

        pos_train = get_files(folder_data, fold_train)
        pos_valid = get_files(folder_data, fold_valid)
        pos_test = get_files(folder_data, fold_test)

        to_h5(folder_data, pos_train, len(pos_train), "fold_"+str(fld_nb)+"_"+"pos_train", outfolder)
        to_h5(folder_data, pos_valid, len(pos_valid), "fold_"+str(fld_nb)+"_"+"pos_valid", outfolder)
        to_h5(folder_data, pos_test, len(pos_test), "fold_"+str(fld_nb)+"_"+"pos_test", outfolder)

        # Load files in each fold then you can use them...